In [4]:
import numpy as np
import pandas as pd
import json

np.random.seed(33)

# Sumber 1: Transaksi (CSV) — 5000 baris
n_trx = 5000
kategori_list = ["Elektronik", "Fashion", "Makanan", "Rumah Tangga", "Kesehatan"]
df_trx_tugas6 = pd.DataFrame({
    "order_id": [f"TX{i}" for i in range(n_trx)],
    "product_id": np.random.randint(1, 31, size=n_trx),
    "unit_terjual": np.random.randint(1, 8, size=n_trx),
    "tanggal": np.random.choice(pd.date_range("2026-10-01","2026-10-31"), size=n_trx).astype(str),
})
df_trx_tugas6.to_csv("tugas6_transaksi.csv", index=False)

# Sumber 2: Data produk (JSON Lines) — 30 produk
produk = [
    {"product_id": i, "nama_produk": f"Produk-{i}", "kategori": np.random.choice(kategori_list),
     "harga": int(np.random.choice([25000,50000,75000,100000,150000,250000]))}
    for i in range(1, 31)
]
with open("tugas6_produk.json", "w") as f:
    for p in produk:
        f.write(json.dumps(p) + "\n")

# Sumber 3: Data ulasan (CSV) — tidak semua transaksi memiliki ulasan (realistis)
n_review = 3500
df_review = pd.DataFrame({
    "order_id": np.random.choice(df_trx_tugas6["order_id"], size=n_review, replace=False),
    "rating": np.random.randint(1, 6, size=n_review),
})
df_review.to_csv("tugas6_ulasan.csv", index=False)

print(f"Tiga sumber data berhasil dibuat:")
print(f"- tugas6_transaksi.csv : {len(df_trx_tugas6)} baris")
print(f"- tugas6_produk.json   : {len(produk)} baris")
print(f"- tugas6_ulasan.csv    : {len(df_review)} baris (tidak seluruh transaksi memiliki ulasan)")

Tiga sumber data berhasil dibuat:
- tugas6_transaksi.csv : 5000 baris
- tugas6_produk.json   : 30 baris
- tugas6_ulasan.csv    : 3500 baris (tidak seluruh transaksi memiliki ulasan)


In [5]:
# A. EXTRACT
df_trx = spark.read.csv("tugas6_transaksi.csv", header=True, inferSchema=True)
df_produk = spark.read.json("tugas6_produk.json")
df_ulasan = spark.read.csv("tugas6_ulasan.csv", header=True, inferSchema=True)

# Menampilkan jumlah baris dan skema masing-masing DataFrame
print(f"Data Transaksi: {df_trx.count()} baris")
df_trx.printSchema()

print(f"Data Produk: {df_produk.count()} baris")
df_produk.printSchema()

print(f"Data Ulasan: {df_ulasan.count()} baris")
df_ulasan.printSchema()

Data Transaksi: 5000 baris
root
 |-- order_id: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- tanggal: timestamp (nullable = true)

Data Produk: 30 baris
root
 |-- harga: long (nullable = true)
 |-- kategori: string (nullable = true)
 |-- nama_produk: string (nullable = true)
 |-- product_id: long (nullable = true)

Data Ulasan: 3500 baris
root
 |-- order_id: string (nullable = true)
 |-- rating: integer (nullable = true)



In [6]:
from pyspark.sql.functions import col, when

# B. TRANSFORM - Penggabungan
# Join transaksi dengan produk, lalu dengan ulasan
df_joined = df_trx.join(df_produk, on="product_id", how="inner") \
                  .join(df_ulasan, on="order_id", how="left")

# Tambahkan kolom total_pendapatan
df_joined = df_joined.withColumn("total_pendapatan", col("unit_terjual") * col("harga"))

# C. TRANSFORM - Penanganan Data Kosong & Pengayaan
# 1. Tambah kolom ada_ulasan SEBELUM fillna (jika rating null = False, selain itu True)
df_joined = df_joined.withColumn("ada_ulasan", when(col("rating").isNull(), False).otherwise(True))

# 2. Isi nilai kosong pada rating dengan 0 menggunakan na.fill()
df_etl = df_joined.na.fill(value=0, subset=["rating"])

# Tampilkan sampel hasil transformasi
df_etl.show(5)

+--------+----------+------------+-------------------+------+------------+-----------+------+----------------+----------+
|order_id|product_id|unit_terjual|            tanggal| harga|    kategori|nama_produk|rating|total_pendapatan|ada_ulasan|
+--------+----------+------------+-------------------+------+------------+-----------+------+----------------+----------+
|     TX0|        21|           2|2026-10-10 00:00:00| 25000|  Elektronik|  Produk-21|     1|           50000|      true|
|     TX1|         8|           6|2026-10-25 00:00:00|250000|     Fashion|   Produk-8|     0|         1500000|     false|
|     TX2|        25|           4|2026-10-13 00:00:00| 25000|  Elektronik|  Produk-25|     2|          100000|      true|
|     TX3|         3|           4|2026-10-18 00:00:00| 75000|     Fashion|   Produk-3|     5|          300000|      true|
|     TX4|        19|           6|2026-10-07 00:00:00| 75000|Rumah Tangga|  Produk-19|     0|          450000|     false|
+--------+----------+---

In [10]:
# D. LOAD
# Sesuaikan 'mahasiswa' dengan username kalian
output_path = "hdfs://localhost:9000/user/mahasiswa/tugas6/hasil_etl"

# Menyimpan ke HDFS, overwrite jika sudah ada, partisi berdasarkan kategori
df_etl.write.mode("overwrite").partitionBy("kategori").parquet(output_path)

# Verifikasi folder di HDFS
!hdfs dfs -ls -R /user/mahasiswa/tugas6/hasil_etl

# Baca kembali dan tampilkan count() sebagai bukti
df_final = spark.read.parquet(output_path)
print("Jumlah baris data hasil ETL yang tersimpan:", df_final.count())

-rw-r--r--   3 yue supergroup          0 2026-09-28 11:01 /user/mahasiswa/tugas6/hasil_etl/_SUCCESS
drwxr-xr-x   - yue supergroup          0 2026-09-28 11:01 /user/mahasiswa/tugas6/hasil_etl/kategori=Elektronik
-rw-r--r--   3 yue supergroup      16364 2026-09-28 11:01 /user/mahasiswa/tugas6/hasil_etl/kategori=Elektronik/part-00000-15d9046a-0d30-44dc-8010-c7c27ae4331b.c000.snappy.parquet
drwxr-xr-x   - yue supergroup          0 2026-09-28 11:01 /user/mahasiswa/tugas6/hasil_etl/kategori=Fashion
-rw-r--r--   3 yue supergroup      11134 2026-09-28 11:01 /user/mahasiswa/tugas6/hasil_etl/kategori=Fashion/part-00000-15d9046a-0d30-44dc-8010-c7c27ae4331b.c000.snappy.parquet
drwxr-xr-x   - yue supergroup          0 2026-09-28 11:01 /user/mahasiswa/tugas6/hasil_etl/kategori=Kesehatan
-rw-r--r--   3 yue supergroup      10393 2026-09-28 11:01 /user/mahasiswa/tugas6/hasil_etl/kategori=Kesehatan/part-00000-15d9046a-0d30-44dc-8010-c7c27ae4331b.c000.snappy.parquet
drwxr-xr-x   - yue supergroup         

In [9]:
from pyspark.sql.functions import avg

# E. Insight Akhir
# Cast boolean ada_ulasan menjadi integer (True=1, False=0) lalu cari rata-ratanya per kategori
df_insight = df_final.withColumn("ada_ulasan_int", col("ada_ulasan").cast("integer"))

df_insight.groupBy("kategori") \
          .agg((avg("ada_ulasan_int") * 100).alias("persentase_ulasan_persen")) \
          .orderBy("persentase_ulasan_persen") \
          .show()

+------------+------------------------+
|    kategori|persentase_ulasan_persen|
+------------+------------------------+
|     Makanan|       68.84328358208955|
|   Kesehatan|       68.88657648283039|
|     Fashion|       70.14492753623188|
|Rumah Tangga|        70.5521472392638|
|  Elektronik|         70.659407138536|
+------------+------------------------+



Mengetahui kategori dengan persentase ulasan terendah memungkinkan tim marketing untuk menargetkan kampanye atau insentif khusus (misalnya: cashback koin jika memberikan review) secara spesifik pada kategori tersebut. Peningkatan jumlah ulasan pada produk yang minim feedback sangat vital karena social proof (bukti sosial) berkorelasi positif dengan peningkatan rasa percaya calon konsumen dan konversi penjualan.